# Pricing Engine

## Score candidates and test the pricing rules

**What.** Convert candidate demand into expected sales, gross profit, feasible prices, and recommendations.

**Why.** Separating these calculations makes the decision trace auditable and lets us test the engine independently of demand estimates.

**How.** Cap sales at stock, apply price bounds and optional hard targets, select the feasible maximum, and test both a binding inventory cap and an infeasible case.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

def score_candidates(rows):
    scored = rows.copy()
    scored["expected_units_sold"] = scored.expected_demand.clip(upper=scored.inventory)
    scored["expected_gross_profit"] = (
        scored.candidate_price - scored.unit_cost
    ) * scored.expected_units_sold
    scored["expected_leftover"] = scored.inventory - scored.expected_units_sold
    scored["feasible"] = (
        scored.candidate_price.between(scored.price_floor, scored.price_ceiling)
        & (scored.hard_unit_target.isna() | scored.expected_units_sold.ge(scored.hard_unit_target))
    )
    return scored

def choose_candidate(rows):
    feasible = rows.loc[rows.feasible]
    if feasible.empty:
        return None
    return feasible.sort_values(
        ["expected_gross_profit", "candidate_price"], ascending=[False, True]
    ).iloc[0]

fixture = pd.DataFrame({
    "candidate_price": [90, 100, 110], "expected_demand": [20, 10, 8],
    "inventory": [11] * 3, "unit_cost": [60] * 3,
    "price_floor": [90] * 3, "price_ceiling": [105] * 3,
    "hard_unit_target": [np.nan] * 3,
})
scored_fixture = score_candidates(fixture)
assert scored_fixture.expected_units_sold.tolist() == [11, 10, 8]
assert scored_fixture.expected_gross_profit.tolist() == [330, 400, 400]
assert choose_candidate(scored_fixture).candidate_price == 100
assert choose_candidate(score_candidates(fixture.assign(hard_unit_target=12))) is None
display(scored_fixture)

   candidate_price  expected_demand  inventory  unit_cost  price_floor  price_ceiling  hard_unit_target  expected_units_sold  expected_gross_profit  expected_leftover  feasible
0               90               20         11         60           90            105               NaN                   11                    330                  0      True
1              100               10         11         60           90            105               NaN                   10                    400                  1      True
2              110                8         11         60           90            105               NaN                    8                    400                  3     False


## Connect the demand forecasts

**What.** Score the candidate demand handoff and compare each recommendation with holding price.

**Why.** The candidate and recommendation tables provide the dashboard with the same underlying results.

**How.** Read the demand export, use the explicit demo cost and price limits from the data preparation cells, and report recommendations and hold-steady deltas by scenario and elasticity assumption.

In [2]:
out = Path("pricewise_outputs")
grid = pd.read_csv(out / "candidate_demand.csv")
assert grid[["unit_cost", "price_floor", "price_ceiling"]].notna().all().all()
grid["cost_and_rules_source"] = "synthetic_scenario"
scored = score_candidates(grid)
key = ["country", "product_id", "segment", "product_type", "week_start", "scenario", "sensitivity"]
results = []
for group_key, rows in scored.groupby(key):
    selected = choose_candidate(rows)
    hold = rows.loc[rows.relative_price.eq(1)].iloc[0]
    result = dict(zip(key, group_key))
    result["status"] = "review" if selected is None else "recommended"
    if selected is not None:
        result.update({
            "current_offer": hold.current_offer, "recommended_price": selected.candidate_price,
            "expected_units_sold": selected.expected_units_sold,
            "expected_gross_profit": selected.expected_gross_profit,
            "units_delta_vs_hold": selected.expected_units_sold - hold.expected_units_sold,
            "profit_delta_vs_hold": selected.expected_gross_profit - hold.expected_gross_profit,
        })
    results.append(result)
recommendations = pd.DataFrame(results)
assert len(recommendations) == scored.groupby(key).ngroups
scored.to_csv(out / "scored_candidates.csv", index=False)
recommendations.to_csv(out / "recommendations.csv", index=False)
display(recommendations.head())

  country  product_id   segment product_type  week_start  scenario sensitivity       status  current_offer  recommended_price  expected_units_sold  expected_gross_profit  units_delta_vs_hold  profit_delta_vs_hold
0      CA  00EX4276FY  Personal     Notebook  2026-08-24  balanced     central  recommended        1739.05             1826.0             0.384814             267.684190            -0.031852             14.071987
1      CA  00EX4276FY  Personal     Notebook  2026-08-24  balanced        high  recommended        1739.05             1652.1             0.538475             280.933397             0.121810             27.321809
2      CA  00EX4276FY  Personal     Notebook  2026-08-24  balanced         low  recommended        1739.05             1826.0             0.410612             285.630193            -0.006054             32.017748
3      CA  00EX4276FY  Personal     Notebook  2026-08-24    excess     central  recommended        1739.05             1826.0             0.384814  

Costs and policy thresholds in the connected example are demo assumptions. A competitor band, joint optimization across shared inventory pools, and production policy approval remain follow-up tasks.